In [ ]:
!pip install ultralytics
!pip install "numpy<2"
! pip install opencv-python

  Using cached matplotlib-3.10.9-cp314-cp314-macosx_11_0_arm64.whl.metadata (52 kB)
  Using cached contourpy-1.3.3-cp314-cp314-macosx_11_0_arm64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached kiwisolver-1.5.0-cp314-cp314-macosx_11_0_arm64.whl.metadata (5.1 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 905.7 kB/s  0:00:01 eta 0:00:01
Using cached matplotlib-3.10.9-cp314-cp314-macosx_11_0_arm64.whl (8.2 MB)
Using cached contourpy-1.3.3-cp314-cp314-macosx_11_0_arm64.whl (273 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 816.2 kB/s  0:00:03 eta 0:00:01
Using cached kiwisolver-1.5.0-cp314-cp314-macosx_11_0_ar

In [ ]:
import psutil

ram = psutil.virtual_memory()
print(f"Total RAM: {ram.total / 1e9:.2f} GB")
print(f"Available RAM: {ram.available / 1e9:.2f} GB")

Total RAM: 135.17 GB
Available RAM: 120.39 GB


In [ ]:
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import shutil   
import torch

In [ ]:
!ls dataset/train/images | head

 IML_Programming_Exam_27_03_2026.ipynb	 data		    datasett.zip
 Introduction_Machine_Learning_Course	 dataset	    yolo_2d.ipynb
 PhenoBench				'dataset (1).zip'


In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────────
MODELS = ["yolov8n-seg.pt", "yolo11n-seg.pt", "yolo26n-seg.pt"]
TRAIN_CFG = dict(
    data="dataset/data.yaml",
    epochs=50,
    imgsz=640,
    batch=10,
    workers=0,
    project="runs/version_compare",
    exist_ok=True,
    # lr0=0.01,     # initial learning rate
    # lrf=0.01,     # final learning rate (lr0 * lrf)
)
OUTPUT_PATH = Path("runs/version_compare/comparison.csv")

# ── Train & collect ────────────────────────────────────────────────────────────
rows = []

for model_name in MODELS:
    tag = model_name.replace(".pt", "")
    print(f"\n{'='*60}\nTraining: {model_name}\n{'='*60}")
    try:
        results = YOLO(model_name).train(**TRAIN_CFG, name=tag)
        df = pd.read_csv(Path(results.save_dir) / "results.csv")
        df.columns = df.columns.str.strip()

        row = {"model": model_name, "status": "success", **df.iloc[-1].to_dict()}
        rows.append(row)
        print(f"{model_name} done")

    except Exception as e:
        rows.append({"model": model_name, "status": f"FAILED: {e}"})
        print(f"{model_name} failed: {e}")

# ── Save ───────────────────────────────────────────────────────────────────────
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
pd.DataFrame(rows).to_csv(OUTPUT_PATH, index=False, encoding="utf-8")
print(f"\n Saved → {OUTPUT_PATH.resolve()}")

## Inference

In [ ]:
model = YOLO("runs/segment/runs/version_compare/yolov8n-seg/weights/best.pt")

results = model.predict(
    source="dataset_split/test/images",  # your test folder
    imgsz=320,
    save=True
)
img = Image.open("runs/segment/predict/phenoBench_00001.jpg")
plt.imshow(img)
plt.axis("off")

## RGB-D FUSTION
RGB-D Fusion for YOLO Segmentation Training

## What is RGB-D Fusion?
RGB-D fusion combines standard RGB (color) images with Depth (D) data to give the model richer spatial information. Instead of the model only seeing color, it also learns **how far objects are** from the camera — which is especially useful for thin or visually similar objects like copper and steel wires.

---

## Why Use RGB-D Fusion?
| RGB Only | RGB-D Fusion |
|---|---|
| Model sees color and texture | Model sees color, texture + distance |
| Thin objects may be missed | Depth edges help distinguish thin objects |
| Objects with similar color are confused | Depth separates objects at different distances |

---

## Data Format
| Data | Format | Location |
|---|---|---|
| RGB images | `.png` / `.jpg` | `dataset_split/{split}/images/` |
| Depth data | `.npy` (raw) or `.png` | `dataset_split/{split}/depth/` |
| Labels | `.txt` (YOLO polygon) | `dataset_split/{split}/labels/` |

> **Why `.npy` over `.png` for depth?**  
> Raw `.npy` files store full 32-bit float precision. Depth `.png` files are quantized to 8-bit (0–255), which loses fine-grained distance detail — especially important for thin objects.

---

## Fusion Strategy: 4-Channel Input (Early Fusion)
We use **early fusion** — depth is added as a 4th channel directly to the image before it enters the model:
This means the model learns RGB and depth **together from the very first layer**, giving it the most complete picture during feature extraction.

### Other fusion strategies (not used here):
| Strategy | How | Why we didn't use it |
|---|---|---|
| Early fusion (ours) | Depth as 4th channel | Best feature learning |
| Late fusion | Two separate models, merge predictions | Double the compute |
| Middle fusion | Merge at intermediate layers | Complex architecture changes |

---

## Depth Normalization
Raw depth values vary widely (e.g. 0–5000mm). We normalize to 0–255 before stacking:

```python
depth_norm = (depth_raw - d_min) / (d_max - d_min + 1e-8)
depth_uint8 = (depth_norm * 255).astype(np.uint8)
```

This preserves relative depth relationships while making the scale compatible with RGB values.

---

## YOLO Architecture Modification
YOLO's first convolutional layer expects **3 channels** by default. We surgically modify only this one layer to accept **4 channels**:

```python
# Original:  Conv2d(in_channels=3, ...)
# Modified:  Conv2d(in_channels=4, ...)
```

### Weight Initialization for the Depth Channel
We don't initialize the depth channel randomly — instead we use the **mean of the RGB weights**:

```python
new_conv.weight[:, :3] = first_conv.weight          # keep pretrained RGB weights
new_conv.weight[:, 3:] = first_conv.weight.mean(dim=1, keepdim=True)  # depth init
```

This way the model starts with strong pretrained RGB knowledge and gradually learns how to use depth during training.

---

## Output Dataset Structure
The fused dataset is saved separately — original data is never modified:
> `test/` is excluded from fusion — it is reserved for inference evaluation after training.

---

## Models Trained
All three models are patched and trained on the fused dataset for direct comparison:

| Model | Architecture | Notes |
|---|---|---|
| `yolov8n-seg` | YOLOv8 nano | Stable baseline |
| `yolo11n-seg` | YOLOv11 nano | Improved over v8 |
| `yolo26n-seg` | YOLOv26 nano | Latest experimental |

---

## Results
Training metrics for all models are saved to:

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────────
DATASET_ROOT = Path("/home/jovyan/work/dataset_split")
FUSED_ROOT   = Path("/home/jovyan/work/dataset_rgbd")
MODELS       = ["yolov8n-seg.pt", "yolo11n-seg.pt", "yolo26n-seg.pt"]
OUTPUT_PATH  = Path("runs/version_compare_rgbd/comparison.csv")
IMG_SIZE     = 640
# ── Fuse RGB + Depth for train and val only ───────────────────────────
def fuse_dataset():
    for split in ["train", "val"]:
        rgb_dir   = DATASET_ROOT / split / "images"
        depth_dir = DATASET_ROOT / split / "depth"
        label_dir = DATASET_ROOT / split / "labels"

        out_img_dir   = FUSED_ROOT / split / "images"
        out_label_dir = FUSED_ROOT / split / "labels"
        out_img_dir.mkdir(parents=True, exist_ok=True)
        out_label_dir.mkdir(parents=True, exist_ok=True)

        skipped, fused = 0, 0

        for rgb_path in sorted([*rgb_dir.glob("*.png"), *rgb_dir.glob("*.jpg")]):
            stem = rgb_path.stem

            # Load depth — npy preferred over png
            depth_npy = depth_dir / f"{stem}.npy"
            depth_png = depth_dir / f"{stem}.png"

            if depth_npy.exists():
                depth_raw = np.load(str(depth_npy)).astype(np.float32)
            elif depth_png.exists():
                depth_raw = cv2.imread(str(depth_png), cv2.IMREAD_ANYDEPTH).astype(np.float32)
            else:
                print(f"No depth for {stem}, skipping")
                skipped += 1
                continue

            # Resize RGB
            rgb = cv2.resize(cv2.imread(str(rgb_path)), (IMG_SIZE, IMG_SIZE))

            # Normalize depth to 0-255
            d_min, d_max = np.nanmin(depth_raw), np.nanmax(depth_raw)
            depth_uint8  = ((depth_raw - d_min) / (d_max - d_min + 1e-8) * 255).astype(np.uint8)
            depth_resized = cv2.resize(depth_uint8, (IMG_SIZE, IMG_SIZE))

            # Save 4-channel fused image
            np.save(str(out_img_dir / f"{stem}.npy"), np.dstack([rgb, depth_resized]))

            # Copy label unchanged
            label_src = label_dir / f"{stem}.txt"
            if label_src.exists():
                shutil.copy(label_src, out_label_dir / f"{stem}.txt")

            fused += 1

        print(f"{split}: {fused} fused, {skipped} skipped")

print("Fusing RGB + Depth...")
fuse_dataset()

In [ ]:
TRAIN_CFG = dict(
    epochs=50,
    imgsz=640,
    batch=10,
    workers=0,
    # lr0=0.01,
    # lrf=0.01,
    project="runs/version_compare_rgbd",
    exist_ok=True,
)

# ──  Patch first conv layer to 4 channels ──────────────────────────────
def patch_model_4ch(model_name: str) -> YOLO:
    model      = YOLO(model_name)
    first_conv = model.model.model[0].conv
    new_conv   = torch.nn.Conv2d(
        in_channels=4,
        out_channels=first_conv.out_channels,
        kernel_size=first_conv.kernel_size,
        stride=first_conv.stride,
        padding=first_conv.padding,
        bias=False
    )
    with torch.no_grad():
        new_conv.weight[:, :3] = first_conv.weight
        new_conv.weight[:, 3:] = first_conv.weight.mean(dim=1, keepdim=True)
    model.model.model[0].conv = new_conv
    print(f"{model_name} patched → 4ch")
    return model

# ── Step 4: Train & collect metrics ───────────────────────────────────────────
rows = []

for model_name in MODELS:
    tag = model_name.replace(".pt", "")
    print(f"\n{'='*60}\nTraining: {model_name}\n{'='*60}")
    try:
        results = patch_model_4ch(model_name).train(
            data="dataset_rgbd/data.yaml",
            name=tag,
            **TRAIN_CFG
        )
        df = pd.read_csv(Path(results.save_dir) / "results.csv")
        df.columns = df.columns.str.strip()
        rows.append({"model": model_name, "status": "success", **df.iloc[-1].to_dict()})
        print(f"{model_name} done")

    except Exception as e:
        rows.append({"model": model_name, "status": f"FAILED: {e}"})
        print(f"{model_name} failed: {e}")

# ── Save comparison CSV ───────────────────────────────────────────────
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
pd.DataFrame(rows).to_csv(OUTPUT_PATH, index=False, encoding="utf-8")
print(f"\n Saved → {OUTPUT_PATH.resolve()}")